# NEXUS MCP explorer

Call the NEXUS MCP server's tools **directly, with no LLM in the loop**, and inspect exactly what comes back.

- Run the cells top to bottom. Change any arguments and re-run.
- The connect cell opens a browser for the SBU login if your saved sign-in has expired.
- Every call is appended to `research/calls.jsonl` (not committed), so you can reload and compare results later.
- `alma_renew_user_loan` and `alma_cancel_user_request` change your real library account. The `call()` helper refuses them unless you pass `allow_write=True`.
- **Before committing this notebook, clear its outputs** (Jupyter: *Clear All Outputs*). Results can include your loans, fines and requests.

## 1. Connect

In [1]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT))
from agent import nexus  # same NEXUS settings, login and saved token as the chat agent

LOG = ROOT / "research" / "calls.jsonl"
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 100)

<@beartype(key_value.aio.stores.base.BaseStore._warn_about_stability) at 0x11a91c680>:13: UserWarning: A configured store is unstable and may change in a backwards incompatible way. Use at your own risk.


In [4]:
await nexus.__aenter__()  # opens the browser for the SBU login if needed
client = nexus.client
tools = await client.list_tools()
TOOLS = {t.name: t for t in tools}
print(f"Connected. {len(tools)} tools.")

Token refresh failed: 401


[11/04/26 11:02:04] INFO     OAuth authorization URL:                                                  ]8;id=10523591;file:///Users/codex/Projects/nexus-testing/.venv/lib/python3.13/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=10523592;file:///Users/codex/Projects/nexus-testing/.venv/lib/python3.13/site-packages/fastmcp/client/auth/oauth.py#408\408]8;;\
                             https://api.clarivate.com/academia/mcp-2/stonybrook/authorize?response_ty             
                             pe=code&client_id=2da2b301-ed12-43aa-acb2-16c8db2fc245&redirect_uri=http%             
                             3A%2F%2Flocalhost%3A8765%2Fcallback&state=MAzCjc6PP6T3REgqRXI73P3sF06RCyX             
                             vKt7XmXINMNk&code_challenge=xNjD0ZGjRr4iz7jMo9r-CT9SdGs-YniWL8tbbiUhcKo&c             
                             ode_challenge_method=S256&resource=https%3A%2F%2Fapi.clarivate.com%2Facad             
                             emia%2Fmcp-2%2Fstonybrook%2Fmcp&scope=openid+profile+offline_access+email             
                             &prompt=consent                                                                       

                    INFO     🎧 OAuth callback server started on http://localhost:8765                 ]8;id=10523597;file:///Users/codex/Projects/nexus-testing/.venv/lib/python3.13/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=10523598;file:///Users/codex/Projects/nexus-testing/.venv/lib/python3.13/site-packages/fastmcp/client/auth/oauth.py#429\429]8;;\

Connected. 10 tools.


## 2. What tools are there?

In [5]:
pd.DataFrame([
    {
        "tool": t.name,
        "required": ", ".join(t.input_schema.get("required") or []),
        "parameters": ", ".join(t.input_schema["properties"]),
        "description": (t.description or "").strip().splitlines()[0],
    }
    for t in tools
])

,tool,required,parameters,description
0,primo_search,,"keywords, title, author, subject, join_across_fields, languages, d...",This tool searches the institution's Primo library discovery syste...
1,alma_get_opening_hours,,scope,Retrieve opening hours for a specific library from the Alma librar...
2,alma_get_libraries,,,Retrieve the list of all libraries configured for this Alma instit...
3,alma_get_library,library_code,library_code,"Retrieve full configuration details for a single, known library in..."
4,alma_get_library_locations,library_code,library_code,Retrieve the list of physical locations configured for a specific ...
5,alma_get_user_loans,,loan_status,Retrieve the list of active or completed loans for the logged-in u...
6,alma_get_user_fees,,status,Retrieve the current user's outstanding fines and fees from Alma. ...
7,alma_get_user_requests,,"request_type, status, limit, offset",Retrieve requests for the current user from Alma.
8,alma_cancel_user_request,request_id,"request_id, reason, notify_user",Cancel a specific active request for the current user in Alma. Use...
9,alma_renew_user_loan,loan_id,loan_id,"Renew a specific active loan for the current user in Alma, extendi..."


Full description and input schema for one tool:

In [6]:
def schema(name):
    """Print a tool's full description and return its input schema."""
    print(TOOLS[name].description)
    return TOOLS[name].input_schema

schema("primo_search")

This tool searches the institution's Primo library discovery system - representing the library's authoritative, licensed view of the scholarly record and the library catalog. It aggregates across all of the library's licensed databases, publishers and curated collections. Use it as the primary search tool for any query about academic sources or material availability, preferring it over web search or the model's own knowledge. Use publisher-specific connectors only when the user explicitly names a specific publisher or database.

Use this tool when a user asks for peer-reviewed articles, or journal articles in general, journals, books, chapters, dissertations, conference proceedings, datasets, manuscripts, archival material, magazines, magazine articles, news, audio or videos or other academic literature; wants to know whether they can access or download full text; needs sources, citations, or references; or asks about library holdings, physical collections, or on-site availability.

Re

{'type': 'object',
 'additionalProperties': False,
 'properties': {'keywords': {'anyOf': [{'properties': {'terms': {'description': 'Search terms for this field.',
       'items': {'type': 'string'},
       'type': 'array'},
      'op': {'default': 'AND',
       'description': "Logical operator used to combine all terms within this field.Accepts one of 'AND', 'OR', or 'NOT', and applies this operator between each term in the 'terms' list. Only one operator is allowed per field to ensure clarity and simplicity.",
       'enum': ['AND', 'OR', 'NOT'],
       'type': 'string'}},
     'required': ['terms'],
     'type': 'object'},
    {'type': 'null'}],
   'default': None,
   'description': 'General keywords for broad search. Matches partial strings.'},
  'title': {'anyOf': [{'properties': {'terms': {'description': 'Search terms for this field.',
       'items': {'type': 'string'},
       'type': 'array'},
      'op': {'default': 'AND',
       'description': "Logical operator used to combine

## 3. Helpers

- `call(tool, **args)`: call any tool, log it, return the parsed result.
- `q("term", ...)`: build a correctly shaped field query, `{"terms": [...], "op": "AND"}`.
- `primo(**args)`: run `primo_search` and return `(table, raw)`: a table of the results **in the order the server returned them**, plus the raw response.
- `compare(a, b)`: compare two result tables (top-1, overlap, rank differences).
- `repeat(n, **args)`: run the same search n times to check whether the server returns the same list every time.

In [41]:
WRITE_TOOLS = {"alma_renew_user_loan", "alma_cancel_user_request"}


async def call(tool, allow_write=False, **args):
    """Call one NEXUS tool directly (no LLM). Returns the parsed result; logs the call to calls.jsonl."""
    if tool in WRITE_TOOLS and not allow_write:
        raise PermissionError(f"{tool} changes your library account. Pass allow_write=True if you mean it.")
    t0 = time.perf_counter()
    r = await client.call_tool(tool, args, raise_on_error=False)
    seconds = round(time.perf_counter() - t0, 3)
    text = "".join(getattr(c, "text", "") for c in r.content)
    try:
        data = json.loads(text)
    except ValueError:
        data = text
    with LOG.open("a") as f:
        f.write(json.dumps({"at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "tool": tool,
                            "args": args, "error": r.is_error, "seconds": seconds, "result": data}, default=str) + "\n")
    print(f"{tool}: {'ERROR' if r.is_error else 'ok'} in {seconds}s")
    if r.is_error:
        print(text)
    return data


def q(*terms, op="AND"):
    """A field query in the shape primo_search requires: q("Silent Spring") -> {"terms": ["Silent Spring"], "op": "AND"}"""
    return {"terms": list(terms)}


COLUMNS = ["Record ID", "Title", "Author", "Edition", "Publication Date", "Physical Format",
           "Holding ID", "Available", "Is physical", "Is electronic", "Delivery category"]


def items_table(data):
    """primo_search items as a table, in server order. sbu_copy = has a Holding ID (Stony Brook owns it)."""
    items = data.get("items", []) if isinstance(data, dict) else []
    df = pd.DataFrame(items)
    if df.empty:
        return df
    df = df[[c for c in COLUMNS if c in df.columns]].copy()
    df.insert(0, "rank", range(1, len(df) + 1))
    df["sbu_copy"] = df["Holding ID"].notna() if "Holding ID" in df else False
    return df.set_index("rank")


async def primo(**args):
    """Run primo_search and return (table, raw response). Shows the totals and the equivalent website search."""
    for key, value in args.items():
        print(f"{key}: {value}")
    data = await call("primo_search", **args)
    if not isinstance(data, dict):
        return pd.DataFrame(), data
    print(f"total={data.get('total')}  page={data.get('page')} of {data.get('pages')}  items on this page={len(data.get('items', []))}")
    if data.get("search_url"):
        display(Markdown(f"[Same search on the library website]({data['search_url']})"))
    return items_table(data), data


def compare(a, b, k=10, names=("A", "B")):
    """Compare the top k of two result tables by Record ID."""
    ids_a, ids_b = list(a["Record ID"][:k]), list(b["Record ID"][:k])
    both = set(ids_a) & set(ids_b)
    print(f"top-1 same: {bool(ids_a and ids_b and ids_a[0] == ids_b[0])}")
    print(f"overlap in top {k}: {len(both)} of {min(len(ids_a), len(ids_b))}")
    titles = {**dict(zip(b["Record ID"], b["Title"])), **dict(zip(a["Record ID"], a["Title"]))}
    rows = [{"Record ID": rid, "Title": titles.get(rid),
             f"rank in {names[0]}": ids_a.index(rid) + 1 if rid in ids_a else None,
             f"rank in {names[1]}": ids_b.index(rid) + 1 if rid in ids_b else None}
            for rid in dict.fromkeys(ids_a + ids_b)]
    return pd.DataFrame(rows).astype({f"rank in {names[0]}": "Int64", f"rank in {names[1]}": "Int64"})


async def repeat(n=3, **args):
    """Run the same primo_search n times; report whether the ordered result list ever changes."""
    runs = []
    for _ in range(n):
        df, _raw = await primo(**args)
        runs.append(list(df["Record ID"]) if not df.empty else [])
    same = all(r == runs[0] for r in runs)
    print(f"\n{n} runs, identical ordered results every time: {same}")
    return runs

## 4. Searches

### Keyword search

In [35]:
q("marine biology")

{'terms': ['marine biology'], 'op': 'AND'}

In [42]:
df, raw = await primo(keywords=q("marine biology"), limit=10)
df

keywords: {'terms': ['marine biology']}
limit: 10
primo_search: ok in 3.247s
total=580068  page=1 of 58007  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28marine%20biology%29)

,Record ID,Title,Author,Edition,Publication Date,Physical Format,Holding ID,Available,Is physical,Is electronic,Delivery category,sbu_copy
rank,,,,,,,,,,,,
1,alma990012036680204856,Marine biology,"Castro, Peter.$$QCastro, Peter.",7th ed.,NaN,book,22382023680004856,NaN,True,False,Alma-P,True
2,cdi_crossref_primary_10_1080_17451000_2026_2612757,Life beneath the waves - Marine Biology Research has sailed into i...,"van der Meij, Sancia E.T. ; Hop, Haakon",NaN,2026,journal,NaN,true,False,False,Remote Search Resource,False
3,alma9917356161104856,Marine biology (Online),NaN,NaN,NaN,journal,22254987500004856,NaN,True,True,Alma-P,True
4,cdi_springer_journals_10_1038_s41579_025_01149_y,Microbial interactions lead to ecosystem function: Marine microbio...,"Vincent, Flora",NaN,2025,journal,NaN,true,False,False,Remote Search Resource,False
5,cdi_wiley_primary_10_1002_rob_22381_ROB22381,A cable‐driven underwater robotic system for delicate manipulation...,"Zarebidoki, Mahmoud ; Dhupia, Jaspreet Singh ; Liarokapis, Minas ;...",NaN,2024,journal,NaN,true,False,False,Remote Search Resource,False
6,cdi_oup_primary_10_1093_nar_gkaa637,Planet Microbe: a platform for marine microbiology to discover and...,"Ponsero, Alise J ; Bomhoff, Matthew ; Blumberg, Kai ; Youens-Clark...",NaN,2021,journal,NaN,true,False,False,Remote Search Resource,False
7,cdi_proquest_miscellaneous_17262222,Ocean biodiversity informatics: a new era in marine biology resear...,"Costello, Mark J. ; Vanden Berghe, Edward",NaN,2006,journal,NaN,true,False,False,Remote Search Resource,False
8,alma990000463590204856,Marine biology,"Thurman, Harold V.$$QThurman, Harold V.",NaN,NaN,book,22381255520004856,NaN,True,False,Alma-P,True
9,alma990005542870204856,Marine biology.,Annual Biology Colloquium (20th : 1959 : Oregon State University)$...,NaN,NaN,book,22381769270004856,NaN,True,False,Alma-P,True


In [33]:
raw.keys()

dict_keys(['items', 'search_url', 'total', 'page', 'pages', 'next_page', 'facets'])

In [34]:
print(raw['search_url'])

https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28marine%20biology%29


### Title + author (a known item)

In [43]:
df_ta, raw_ta = await primo(title=q("Silent Spring"), author=q("Carson"), limit=10)
df_ta

title: {'terms': ['Silent Spring']}
author: {'terms': ['Carson']}
limit: 10
primo_search: ok in 3.731s
total=50  page=1 of 5  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=title%2Ccontains%2C%28Silent%20Spring%29%2CAND%3Bcreator%2Ccontains%2C%28Carson%29)

,Record ID,Title,Author,Edition,Publication Date,Physical Format,Holding ID,Available,Is physical,Is electronic,Delivery category,sbu_copy
rank,,,,,,,,,,,,
1,alma990005212040204856,Silent spring,"Carson, Rachel, 1907-1964, author.$$QCarson, Rachel",NaN,NaN,book,22381715860004856,NaN,True,False,Alma-P,True
2,cdi_proquest_miscellaneous_59904830,Silent Spring,"Cooper, Richard N.",NaN,1997,journal,NaN,true,False,False,Remote Search Resource,False
3,alma990011903650204856,Silent spring,"Carson, Rachel, 1907-1964, author.$$QCarson, Rachel",First Mariner books edition.,NaN,book,22267643770004856,NaN,True,False,Alma-P,True
4,cdi_jstor_primary_10_2307_2257609,Silent Spring,"Davis, B. N. K.",NaN,1964,journal,NaN,true,False,False,Remote Search Resource,False
5,cdi_crossref_primary_10_2307_1292913,Silent Spring,"Went, Frits",NaN,1963,journal,NaN,true,False,False,Remote Search Resource,False
6,cdi_jstor_primary_10_2307_1441323,Silent Spring,"Ferguson, Denzel E.",NaN,1963,journal,NaN,true,False,False,Remote Search Resource,False
7,cdi_gale_onefilemisc_PPPM_A18246331,Silent spring: the obligation to endure,"Carson, Rachel",NaN,1996,journal,NaN,true,False,False,Remote Search Resource,False
8,alma991028889509704801,Silent spring,"Carson, Rachel, 1907-1964.$$QCarson, Rachel",25th anniversary ed.,NaN,book,NaN,no_inventory,True,False,Alma-P,False
9,alma990012355770204856,A sense of wonder : two interviews with Rachel Carson,NaN,NaN,NaN,book,22383267110004856,NaN,True,False,Alma-P,True


### A wrongly shaped call

This is the kind of call gemma4 kept sending: the title as a plain string. The server rejects it, and the error text below is exactly what the model sees.

In [44]:
await call("primo_search", title="Silent Spring", author="Rachel Carson")

primo_search: ERROR in 1.198s
2 validation errors for call[search]
title
  Input should be a valid dictionary or instance of FieldQuery [type=model_type, input_value='Silent Spring', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type
author
  Input should be a valid dictionary or instance of FieldQuery [type=model_type, input_value='Rachel Carson', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type Error ID: 8def305a-0d5d-4f8e-bf59-907c54e76819.


"2 validation errors for call[search]\ntitle\n  Input should be a valid dictionary or instance of FieldQuery [type=model_type, input_value='Silent Spring', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/model_type\nauthor\n  Input should be a valid dictionary or instance of FieldQuery [type=model_type, input_value='Rachel Carson', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/model_type Error ID: 8def305a-0d5d-4f8e-bf59-907c54e76819."

### Filters and sorting

Try `resources_types`, `sort` (`relevance`, `newest`, `oldest`, `title`, `author`), `languages`, `online_resources`, `peer_reviewed`, `page`.

In [13]:
df_new, raw_new = await primo(keywords=q("CRISPR"), resources_types=["books"], sort="newest", limit=10)
df_new

primo_search: ok in 2.841s
total=244  page=1 of 25  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=date_d&tab=Everything&scope=EverythingNZBooks&multiFacets=facet_rtype%2Cinclude%2Cbooks&query=any%2Ccontains%2C%28CRISPR%29)

,Record ID,Title,Author,Edition,Publication Date,Physical Format,Holding ID,Available,Is physical,Is electronic,Delivery category,sbu_copy
rank,,,,,,,,,,,,
1,alma991034443059704801,CRISPR : methods and protocols,NaN,NaN,NaN,book,NaN,no_inventory,True,False,Alma-P,False
2,cdi_nas_primary_23405,"Gene Drives on the Horizon: Advancing Science, Navigating Uncertai...","National Academies of Sciences, Engineering, and Medicine ; Studie...",1,2016,book,NaN,true,False,False,Remote Search Resource,False
3,alma991061708308604801,CRISPR : a powerful way to change DNA,"Ridge, Yolanda, 1973- author.$$QRidge, Yolanda",NaN,NaN,book,NaN,no_inventory,True,False,Alma-P,False
4,cdi_casalini_monographs_5494953,Genome Editing in Neurosciences,"Jaenisch, Rudolf ; Zhang, Feng ; Gage, Fred",1,2017,book,NaN,true,False,False,Remote Search Resource,False
5,alma990017097280204856,CRISPR-Cas : a laboratory manual,NaN,NaN,NaN,book,22382363310004856,NaN,True,False,Alma-P,True
6,cdi_proquest_ebookcentral_EBC6676091,RNA Damage and Repair,"Kotta-Loizou, Ioly",1st Edition 2021,2021,book,NaN,true,False,False,Remote Search Resource,False
7,cdi_proquest_ebookcentral_EBC7260882,Systems biology : functional strategies of living organisms,"Konieczny, Leszek ; Roterman-Konieczna, Irena ; Spólnik, Pawel",2,2023,book,NaN,true,False,False,Remote Search Resource,False
8,cdi_springer_books_10_1007_978_1_0716_0294_2,Trypanosomatids: Methods and Protocols,"Michels, Paul A. M ; Ginger, Michael L ; Zilberstein, Dan",1st Edition 2020,2020,book,NaN,true,False,False,Remote Search Resource,False
9,alma9917522929304856,CRISPR-Cas enzymes,NaN,NaN,NaN,book,NaN,Not available,False,True,Alma-E,False


## 5. Compare two searches

Same book, two ways of asking: keywords versus title and author fields.

In [14]:
df_kw, _ = await primo(keywords=q("Silent Spring Rachel Carson"), limit=10)
compare(df_kw, df_ta, names=("keywords", "title+author"))

primo_search: ok in 3.447s
total=821  page=1 of 83  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28Silent%20Spring%20Rachel%20Carson%29)

top-1 same: False
overlap in top 10: 2 of 10


,Record ID,Title,rank in keywords,rank in title+author
0,alma990011903650204856,Silent spring,1,3
1,cdi_crossref_primary_10_1080_23311983_2026_2700915,Making slow violence legible: Environmental justice in Rachel Cars...,2,<NA>
2,alma990005212040204856,Silent spring,3,1
3,cdi_jstor_primary_26360868,Corporate Chemistry: A Biopolitics of Environment in Rachel Carson...,4,<NA>
4,cdi_crossref_primary_10_3197_096734022X16627150607989,Reimagining Just Futures with Rachel Carson's Silent Spring,5,<NA>
5,cdi_sage_journals_10_1177_0090591717712024,Queer/Love/Bird Extinction: Rachel Carson's Silent Spring as a Wor...,6,<NA>
6,cdi_crossref_primary_10_2478_host_2023_0013,Circulating Silence. The Reaction to Rachel Carson’s Book Silent S...,7,<NA>
7,alma991001569929704801,Silent spring,8,<NA>
8,alma991004484549704801,Silent spring,9,<NA>
9,alma990000174380304811,Silent spring.,10,<NA>


## 6. Is the server consistent?

The same search several times. If the order changes between identical calls, small differences elsewhere are noise.

In [15]:
runs = await repeat(3, keywords=q("marine biology"), limit=10)

primo_search: ok in 3.465s
total=580068  page=1 of 58007  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28marine%20biology%29)

primo_search: ok in 1.538s
total=580068  page=1 of 58007  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28marine%20biology%29)

primo_search: ok in 0.989s
total=580068  page=1 of 58007  items on this page=10


[Same search on the library website](https://search.library.stonybrook.edu/discovery/search?vid=01SUNY_STB%3A01SUNY_STB&limit=10&offset=0&sort=rank&tab=Everything&scope=EverythingNZBooks&query=any%2Ccontains%2C%28marine%20biology%29)


3 runs, identical ordered results every time: True


## 7. Look at one record in full

Everything the server returns for a single result, including fields the table leaves out.

In [16]:
raw_ta["items"][0]

{'Record ID': 'alma990005212040204856',
 'MMS ID': '990005212040204856',
 'Title': 'Silent spring ',
 'Author': 'Carson, Rachel, 1907-1964, author.$$QCarson, Rachel',
 'Additional Author': 'Darling, Lois, illustrator.$$QDarling, Lois; Darling, Louis, illustrator.$$QDarling, Louis; Houghton Mifflin Company, publisher.$$QHoughton Mifflin Company; Riverside Press (Cambridge, Mass.), printer.$$QRiverside Press (Cambridge, Mass.)',
 'Edition': None,
 'Journal, Series': None,
 'Language': 'eng',
 'Open Access Indication': None,
 'Usage Rights': None,
 'Peer Reviewed': None,
 'Abstract': '"For as long as man has dwelt on this planet, spring has been the season of rebirth, and the singing of birds. Now in some parts of America spring is strangely silent, for many of the birds are dead - incidental victims of our reckless attempt to control our environment by the use of chemicals that poison not only the insects against which they are directed but the birds in the air, the fish in the rivers, t

### Facets

Counts for the whole result set, not per item.

In [17]:
for facet in raw_ta.get("facets", []):
    for name, values in facet.items():
        print(f"{name}: {values[:8]}")

rtype: ['reviews (6)', 'magazinearticle (2)', 'articles (1)', 'videos (1)', 'books (39)', 'audiobook (1)', 'audios (1)', 'scores (1)']
institution: ['4830 (7)', '4844 (11)', '4843 (9)', '4808 (6)', '4806 (6)', '4805 (6)', '4804 (7)', '4803 (11)']
creationdate: ['1961 (1)', '1963 (3)', '1964 (1)', '1996 (1)', '1997 (1)', '2000 (2)', '2006 (1)', '1955 (1)']
creator: ['Darling Lois (4)', 'Gordon Isabel S (1)', 'Burnside John (1)', 'Foley Martha (2)', 'Gancher David (1)', 'Brooks Gwendolyn (1)', 'Fundação Calouste Gulbenkian (1)', 'Lear Linda J (2)']
tlevel: ['peer_reviewed (3)', 'online_resources (9)', 'available_p (11)']
library: ['4856–123069210004856 (8)', '4856–1537530340004856 (3)', '4856–123074070004856 (1)']
location_code: ['4856–123069210004856–main (7)', '4856–1537530340004856–sci (3)', '4856–123069210004856–av (1)', '4856–123074070004856–spec (1)']
domain: ['JSTOR Archival Journals and Primary Sources Collection (6)', 'Jstor Complete Legacy (6)', 'ProQuest One Academic (New) (5)

## 8. Other tools

Library information:

In [18]:
libraries = await call("alma_get_libraries")
libraries

alma_get_libraries: ok in 2.21s


[{'code': 'STBFL',
  'name': '4th Floor Storage',
  'description': None,
  'locations_count': 5,
  'circ_desks_count': 1,
  'campus_info': 'STBWEST'},
 {'code': 'STBAR',
  'name': 'Archives',
  'description': None,
  'locations_count': 2,
  'circ_desks_count': 1,
  'campus_info': 'STBWEST'},
 {'code': 'STBBA',
  'name': 'Bibliographic Access Svcs',
  'description': None,
  'locations_count': 2,
  'circ_desks_count': 1,
  'campus_info': 'STBWEST'},
 {'code': 'STB-BRS',
  'name': 'Borrowing Resource Sharing Library',
  'description': None,
  'locations_count': 1,
  'circ_desks_count': 1,
  'campus_info': ''},
 {'code': 'STBCR',
  'name': 'Central Reading Room',
  'description': None,
  'locations_count': 6,
  'circ_desks_count': 1,
  'campus_info': 'STBWEST'},
 {'code': 'STBCH',
  'name': 'Chemistry Library',
  'description': None,
  'locations_count': 9,
  'circ_desks_count': 1,
  'campus_info': 'STBWEST'},
 {'code': 'ELEC',
  'name': 'ELEC',
  'description': None,
  'locations_count': 

In [19]:
await call("alma_get_opening_hours", scope="STBMN")  # STBMN = Main Stacks; codes come from alma_get_libraries

alma_get_opening_hours: ok in 1.822s


[{'date': '2026-11-09',
  'day_of_week': 'Monday',
  'description': 'Open 08:30-20:00',
  'status': 'Open',
  'hours': [{'from_time': '08:30', 'to_time': '20:00'}]},
 {'date': '2026-11-06',
  'day_of_week': 'Friday',
  'description': 'Open 08:30-17:00',
  'status': 'Open',
  'hours': [{'from_time': '08:30', 'to_time': '17:00'}]},
 {'date': '2026-11-07',
  'day_of_week': 'Saturday',
  'description': 'Open 12:00-17:00',
  'status': 'Open',
  'hours': [{'from_time': '12:00', 'to_time': '17:00'}]},
 {'date': '2026-11-08',
  'day_of_week': 'Sunday',
  'description': 'Open 12:00-17:00',
  'status': 'Open',
  'hours': [{'from_time': '12:00', 'to_time': '17:00'}]},
 {'date': '2026-11-05',
  'day_of_week': 'Thursday',
  'description': 'Open 08:30-20:00',
  'status': 'Open',
  'hours': [{'from_time': '08:30', 'to_time': '20:00'}]},
 {'date': '2026-11-04',
  'day_of_week': 'Wednesday',
  'description': 'Open 08:30-20:00',
  'status': 'Open',
  'hours': [{'from_time': '08:30', 'to_time': '20:00'}]

Your own account (personal data: clear these outputs before committing):

In [28]:
await call("alma_get_user_loans", loan_status="Active")

alma_get_user_loans: ok in 1.489s


{'total_loans': 1,
 'loans': [{'loan_id': '49810984020004856',
   'item_barcode': '31794021554085',
   'title': 'The Cambridge illustrated dictionary of natural history / R.J. Lincoln and G.A. Boxshall ; illustrations by Roberta Smith.',
   'author': 'Lincoln, Roger J.',
   'description': None,
   'publication_year': '1987.',
   'library_code': 'STBNR',
   'library_name': 'North Reading Room',
   'location_code': 'SCI',
   'location_name': 'SCI',
   'call_number': 'QH13 .L56 1987',
   'circ_desk': 'NRR_CIRC',
   'loan_date': '2026-11-04T16:12:27.773Z',
   'due_date': '2026-12-18T04:59:00Z',
   'return_date': None,
   'loan_status': 'ACTIVE',
   'process_status': 'NORMAL',
   'renewable': True,
   'loan_fine': None,
   'mms_id': '990007379590204856',
   'holding_id': '22382140780004856',
   'item_id': '23273793300004856',
   'item_policy': '01',
   'last_renew_date': None,
   'last_renew_status': '',
   'returned_by': None,
   'request_id': None}]}

In [21]:
await call("alma_get_user_fees")

alma_get_user_fees: ok in 1.928s


{'total_sum': 0.0, 'fees': []}

## 9. Reload the call log

Every call made from this notebook, across sessions.

In [22]:
log = pd.read_json(LOG, lines=True)
log[["at", "tool", "args", "error", "seconds"]].tail(20)

,at,tool,args,error,seconds
0,2026-11-04T16:03:53+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,2.896
1,2026-11-04T16:05:00+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,7.636
2,2026-11-04T16:05:03+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,2.691
3,2026-11-04T16:05:06+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,3.277
4,2026-11-04T16:06:05+00:00,primo_search,"{'title': {'terms': ['Silent Spring'], 'op': 'AND'}, 'author': {'t...",False,2.664
5,2026-11-04T16:06:19+00:00,primo_search,"{'title': 'Silent Spring', 'author': 'Rachel Carson'}",True,1.608
6,2026-11-04T16:06:39+00:00,primo_search,"{'keywords': {'terms': ['CRISPR'], 'op': 'AND'}, 'resources_types'...",False,2.841
7,2026-11-04T16:06:55+00:00,primo_search,"{'keywords': {'terms': ['Silent Spring Rachel Carson'], 'op': 'AND...",False,3.447
8,2026-11-04T16:07:55+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,3.465
9,2026-11-04T16:07:57+00:00,primo_search,"{'keywords': {'terms': ['marine biology'], 'op': 'AND'}, 'limit': 10}",False,1.538


## 10. Disconnect

In [ ]:
await nexus.__aexit__(None, None, None)
print("Disconnected.")